In [0]:
%sql
DROP TABLE IF EXISTS orders_ext;
CREATE OR REPLACE TABLE orders_ext (
order_id BIGINT,
sku STRING,
product_name STRING,
product_category STRING,
qty INT,
unit_price DECIMAL(10,2)
) using delta
LOCATION 'abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat1'

In [0]:
from decimal import Decimal
import random

categories = ["Electronics", "Clothing", "Books", "Food", "Furniture"]
products = [
    ("SKU001", "Laptop", "Electronics"),
    ("SKU002", "T-Shirt", "Clothing"),
    ("SKU003", "Python Book", "Books"),
    ("SKU004", "Coffee Beans", "Food"),
    ("SKU005", "Office Chair", "Furniture"),
]

rows = []
for i in range(1, 25):
    sku, name, category = random.choice(products)
    qty = random.randint(1, 50)
    unit_price = round(random.uniform(5.0, 500.0), 2)
    rows.append((i, sku, name, category, qty, Decimal(str(unit_price))))

df = spark.createDataFrame(rows, schema="order_id BIGINT, sku STRING, product_name STRING, product_category STRING, qty INT, unit_price DECIMAL(10,2)")
df.write.mode("append").saveAsTable("orders_ext")


In [0]:
%sql
describe history orders_ext

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-08-31T07:18:13Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(472239495099700),0826-063441-ydade36q,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 24, numOutputBytes -> 2796)",null,Databricks-Runtime/16.4.x-scala2.13
1,2026-08-31T07:13:15Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(472239495099700),0826-063441-ydade36q,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 24, numOutputBytes -> 2751)",null,Databricks-Runtime/16.4.x-scala2.13
0,2026-08-31T07:07:21Z,147836707444603,anooptu@gmail.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(472239495099700),0826-063441-ydade36q,null,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13


In [0]:
%sql
select * from orders_ext

order_id,sku,product_name,product_category,qty,unit_price
1,SKU001,Laptop,Electronics,13,409.50
2,SKU004,Coffee Beans,Food,19,384.96
3,SKU005,Office Chair,Furniture,25,169.16
4,SKU005,Office Chair,Furniture,8,10.46
5,SKU003,Python Book,Books,34,466.29
6,SKU003,Python Book,Books,35,327.43
7,SKU001,Laptop,Electronics,43,404.58
8,SKU005,Office Chair,Furniture,49,402.26
9,SKU003,Python Book,Books,12,387.84
10,SKU001,Laptop,Electronics,24,423.22


PARITITIONED TABLE

In [0]:
%sql
DROP TABLE IF EXISTS orders_ext_part;
CREATE OR REPLACE TABLE orders_ext_part (
order_id BIGINT,
sku STRING,
product_name STRING,
product_category STRING,
qty INT,
unit_price DECIMAL(10,2)
) using delta
PARTITIONED BY (product_category)
LOCATION 'abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2'


In [0]:
%sql
insert into orders_ext_part select * from orders_ext

num_affected_rows,num_inserted_rows
48,48


In [0]:
from decimal import Decimal
import random

categories = ["Electronics", "Clothing", "Books", "Food", "Furniture"]
products = [
    ("SKU001", "Laptop", "Electronics"),
    ("SKU002", "T-Shirt", "Clothing"),
    ("SKU003", "Python Book", "Books"),
    ("SKU004", "Coffee Beans", "Food"),
    ("SKU005", "Office Chair", "Furniture"),
]

rows = []
for i in range(1, 10):
    sku, name, category = random.choice(products)
    qty = random.randint(1, 50)
    unit_price = round(random.uniform(5.0, 500.0), 2)
    rows.append((i, sku, name, category, qty, Decimal(str(unit_price))))

df = spark.createDataFrame(rows, schema="order_id BIGINT, sku STRING, product_name STRING, product_category STRING, qty INT, unit_price DECIMAL(10,2)")
df.write.mode("append").saveAsTable("orders_ext_part")


In [0]:
from decimal import Decimal
import random

categories = ["Electronics", "Clothing", "Books", "Food", "Furniture"]
products = [
    ("SKU001", "Laptop", "Electronics"),
    ("SKU002", "T-Shirt", "Clothing"),
    ("SKU003", "Python Book", "Books"),
    ("SKU004", "Coffee Beans", "Food"),
    ("SKU005", "Office Chair", "Furniture"),
]

rows = []
for i in range(1, 10):
    sku, name, category = random.choice(products)
    qty = random.randint(1, 50)
    unit_price = round(random.uniform(5.0, 500.0), 2)
    rows.append((i, sku, name, category, qty, Decimal(str(unit_price))))

df = spark.createDataFrame(rows, schema="order_id BIGINT, sku STRING, product_name STRING, product_category STRING, qty INT, unit_price DECIMAL(10,2)")
df.write.mode("append").saveAsTable("orders_ext_part")


In [0]:
%sql
describe detail orders_ext_part

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,195dc97b-a3b8-4efb-9697-ffed63f7b874,useastws.default.orders_ext_part,null,abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2,2026-08-31T07:16:39.267Z,2026-08-31T07:36:49Z,List(product_category),List(),14,32683,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
select distinct _metadata.file_path from orders_ext_part;

file_path
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2/product_category=Food/part-00001-5ddff6c1-3932-4ddc-99e8-80d0eee3a3db.c000.snappy.parquet
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2/product_category=Electronics/part-00004-de41bd38-570f-452c-a46d-74e1bbc60599.c000.snappy.parquet
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2/product_category=Furniture/part-00000-e749cb30-f732-4db4-a206-b095d22d87e7.c000.snappy.parquet
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2/product_category=Books/part-00002-be5f93e2-e6e7-457b-ab33-de0b672a6aa9.c000.snappy.parquet
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2/product_category=Books/part-00002-e8257ad9-db46-4ccc-9986-2fc84aaa4a23.c000.snappy.parquet
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2/product_category=Clothing/part-00003-d8d969fb-e311-401e-bb0b-6517f73219d7.c000.snappy.parquet
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2/product_category=Clothing/part-00003-278c6be8-33c5-4341-b4cb-92c433c88e68.c000.snappy.parquet
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2/product_category=Furniture/part-00000-82b185ec-738e-44f0-802f-dc22392e5adc.c000.snappy.parquet
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2/product_category=Food/part-00001-c0938fe9-d612-4033-b75c-341c9b8d3246.c000.snappy.parquet
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2/product_category=Clothing/part-00002-e2b70904-6022-4fd7-8bba-8ccb86602375.c000.snappy.parquet


In [0]:
%sql
describe history orders_ext_part

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-08-31T07:36:49Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(472239495099700),0826-063441-ydade36q,2,WriteSerializable,true,"Map(numFiles -> 4, numOutputRows -> 9, numOutputBytes -> 9264)",null,Databricks-Runtime/16.4.x-scala2.13
2,2026-08-31T07:36:07Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(472239495099700),0826-063441-ydade36q,1,WriteSerializable,true,"Map(numFiles -> 5, numOutputRows -> 9, numOutputBytes -> 11273)",null,Databricks-Runtime/16.4.x-scala2.13
1,2026-08-31T07:18:45Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(472239495099700),0826-063441-ydade36q,0,WriteSerializable,true,"Map(numFiles -> 5, numOutputRows -> 48, numOutputBytes -> 12146)",null,Databricks-Runtime/16.4.x-scala2.13
0,2026-08-31T07:16:39Z,147836707444603,anooptu@gmail.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [""product_category""], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(472239495099700),0826-063441-ydade36q,null,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13


In [0]:
%sql
show partitions orders_ext_part

product_category
Food
Electronics
Clothing
Books
Furniture


In [0]:
%sql
select product_category,  sum(qty) as total_qty
from orders_ext_part where product_category = 'Food'
 group by product_category

product_category,total_qty
Food,420


From Spark UI for above query
- number of files pruned	11 
- number of files read	3  
- number of parquet row groups read	3 
- number of partition columns	1  
- number of partitions read	1  
- number of scanned columns	2

In [0]:
%sql
select count(*) from orders_ext_part

count(1)
66


No files were read for select count(*).. as the count was present in metadata itself. No need of manually counting rows

In [0]:
%sql
select product_category,  sum(qty) as total_qty
from orders_ext_part where unit_price >100 
 group by product_category

product_category,total_qty
Food,420
Electronics,391
Books,272
Furniture,275
Clothing,269


From spark UI for above query. Since query doesnt use partition column in where clause, no benefits
- number of files pruned	0
- number of files read	14
- number of parquet row groups read	14
- number of partition columns	1
- number of partitions read	5
- number of scanned columns	3

In [0]:
%sql
optimize orders_ext_part

path,metrics
abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat2,"List(5, 14, List(2464, 2554, 2503.6, 5, 12518), List(2200, 2483, 2334.5, 14, 32683), 5, null, null, 0, 1, 14, 0, true, 0, 0, 1788162754972, 1788162759885, 4, 5, null, List(0, 0), null, 6, 6, 1435, 0, null)"


In [0]:
%sql
select product_category,  sum(qty) as total_qty
from orders_ext_part where unit_price >100 
 group by product_category

product_category,total_qty
Electronics,391
Furniture,275
Food,420
Books,272
Clothing,269


After OPTIMIZE data files where combined inside each partition and hence file read is reduced
- number of files pruned	0
- number of files read	5
- number of parquet row groups read	5
- number of partition columns	1
- number of partitions read	5
- number of scanned columns	3

- Partitions can be done only on columns with low cardinality else its not benefitial
- No obvious benefits for data skipping. 
- If usage pattern changes, partitioning need to be changed
- To change partitioning, whole table needs to be re written
- Can create small file problems
- Can lead to skewed data
